# 🌡️ Climate Trend Analyzer — Walkthrough Notebook
This notebook walks through the full pipeline step by step, so you can **see** every intermediate result.
The heavy lifting lives in `src/` (reusable modules); the notebook just calls them and explains the output.

**Run order:** Kernel → Restart & Run All. Takes ~1–2 minutes.

In [ ]:
import sys, os
from pathlib import Path
# Make the project root importable when the notebook runs from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from src import config
pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)

## Step 1 — Virtual simulation: generate raw climate data

In [ ]:
from src.data_generator import generate_climate_data
raw = generate_climate_data()
raw.head(10)

In [ ]:
print("Shape:", raw.shape)
print("Missing values per column:")
print(raw.isna().sum())
print("Sensor error codes (-999/999):", int(raw.select_dtypes("number").isin([-999, 999]).sum().sum()))
print("Duplicate (city, date) rows:", raw.duplicated(subset=["city", "date"]).sum())

In [ ]:
# Ground truth: the extreme events we injected (lets us evaluate detection later)
pd.read_csv(config.INJECTED_EVENTS_FILE).head(10)

## Step 2 — Data cleaning

In [ ]:
from src.preprocessing import clean_data
clean = clean_data(raw)
pd.read_csv(config.TABLES_DIR / "data_quality_report.csv")

## Step 3 — Feature engineering (climatology, anomalies, seasons)

In [ ]:
from src.feature_engineering import run_feature_engineering
df, monthly, yearly = run_feature_engineering(clean)
df[["date","city","temp_mean","temp_normal","temp_anomaly","temp_zscore","season","hot_day","rainy_day"]].sample(8, random_state=1)

## Step 4 — Exploratory Data Analysis

In [ ]:
from src.eda import run_eda
stats = run_eda(df)
stats

In [ ]:
display(Image(config.FIGURES_DIR / "03_seasonal_cycle.png"))

**What to notice:** Delhi has a large summer–winter swing (continental climate); Mumbai and Chennai are maritime and stable.
Chennai's rainfall peaks in **Oct–Nov** (north-east monsoon) while the other cities peak in **Jun–Sep** (south-west monsoon).

## Step 5 — Long-term trend analysis (OLS, Mann-Kendall, Sen's slope)

In [ ]:
from src.trend_analysis import run_trend_analysis
summary, decadal, seasonal = run_trend_analysis(df, monthly, yearly)
summary[summary.variable == "temp_mean"][["city","sen_slope_per_decade","ols_slope_per_decade","mk_p_value","trend","change_last_vs_first_decade"]]

In [ ]:
display(Image(config.FIGURES_DIR / "06_temperature_trend.png"))
display(Image(config.FIGURES_DIR / "08_warming_stripes.png"))

In [ ]:
# Did we recover the warming rate that was injected into the simulation?
true = pd.Series({c: p["warming_per_decade"] for c, p in config.CITY_PROFILES.items()}, name="injected")
est = summary[summary.variable=="temp_mean"].set_index("city")["sen_slope_per_decade"].rename("estimated")
pd.concat([true, est], axis=1).round(3)

In [ ]:
summary[summary.variable == "rainfall_total"][["city","sen_slope_per_decade","mk_p_value","trend"]]

## Step 6 — Anomaly detection

In [ ]:
from src.anomaly_detection import run_anomaly_detection
df_flags, events, extreme_rain, monthly_flags, evaluation = run_anomaly_detection(df, monthly)
events.sort_values("peak_zscore", key=abs, ascending=False).head(10)

In [ ]:
evaluation   # recall/precision against the injected ground truth

In [ ]:
display(Image(config.FIGURES_DIR / "14_temperature_anomalies.png"))

## Step 7 — Forecasting (back-test, then 5-year forecast)

In [ ]:
from src.forecasting import run_forecasting
metrics, forecast = run_forecasting(monthly)
metrics.pivot(index="city", columns="model", values="RMSE")

In [ ]:
display(Image(config.FIGURES_DIR / "18_temperature_forecast.png"))

## Step 8 — Automatic insights report

In [ ]:
from src.report_generator import generate_report, build_insights
generate_report()
for i in build_insights():
    print(f"[{i['category']}] {i['insight']}")

## ✅ Done
- Figures → `outputs/figures/`  ·  Tables → `outputs/tables/`  ·  Report → `reports/climate_insights_report.md`
- Interactive dashboard: `streamlit run app/streamlit_app.py`